# ENGR 1451/2451 — Weekly Assignment 7
**10 points · 4 graded parts**  
**Due:** See Canvas.

Practice confounding, standard error, and confidence intervals. Show your Python code and keep written answers brief.


## What to hand in

Work with the **course tutor** in the class project. Use it as your AI help; it keeps the record, so no separate log is needed. Your code and written answers must still be your own.

At the end of each working session, type **"make my record"** in the chat. Copy its block into **Session records** below, placing each new record after the previous one.

To continue later, start a new chat in the same project and state the assignment and where you stopped. Session records are not graded for correctness or how much help you needed.


## Setup

All data are embedded; no additional files are needed. Keep full precision in calculations and round only displayed results.


In [21]:
import numpy as np
import pandas as pd
import scipy.stats as stats

pd.set_option("display.precision", 4)


## LE7-1. Group composition and confounding (4 pts)

These synthetic inspection records compare two manufacturing methods. Methods were **not randomly assigned** to parts.

Calculate pass rates for each method within each part type and overall. For overall rates, use `groupby()` and sum the counts before dividing; do not average percentages.

Which method has the higher overall rate? Which has the higher rate within each part type? In two sentences, explain the reversal using group composition and why these observations do not establish a causal effect of method.

**Answer:** Method B has the higher overall rate; however method A has a higher pass rate within simple and complex parts. This reverse effect is illustrative of Simpson's Paradox. Simple parts from this sample have a higher pass rate than complex parts, and since more simple parts were inspected using method B than method A (and more complex parts were inspected using method A than method B), the overall pass rate for method B is higher than method A. These observations do not establish a causal effect of method because the experiment did not involve random assignment of methods to parts, thus resulting in Simpson's Paradox being present in the data.


In [22]:
inspection = pd.DataFrame({
    "method": ["A", "A", "B", "B"],
    "part_type": ["Simple", "Complex", "Simple", "Complex"],
    "inspected": [20, 80, 80, 20],
    "passed": [18, 48, 64, 10],
})

# Calculate and display pass rates within each part type.
# Group by method, sum inspected/passed counts, and calculate overall rates.
pass_rates = inspection.groupby("method")["passed"].sum()/inspection.groupby("method")["inspected"].sum()
specific_pass_rates = inspection["passed"]/inspection["inspected"]
print("Pass rate (%) for method A:",pass_rates["A"]*100,
"\nPass rate (%) for method B: ",round(pass_rates["B"]*100,2))

inspection_complete = pd.concat([inspection,specific_pass_rates], axis=1, keys = ["Inspection Data"] + ["Pass Rate"])
inspection_complete.head()


Pass rate (%) for method A: 66.0 
Pass rate (%) for method B:  74.0


Inspection Data                            Pass Rate
           method part_type inspected passed         0
0               A    Simple        20     18       0.9
1               A   Complex        80     48       0.6
2               B    Simple        80     64       0.8
3               B   Complex        20     10       0.5

## LE7-2. Estimate a population mean (4 pts)

The synthetic tensile strengths below are in **MPa**. Assume an independent random sample from a stable, approximately normal production process. Its population mean and SD are unknown.

### A. Estimate the SD and SE (2 pts)

Display $n$, $\bar x$, sample SD $s$ using `std(ddof=1)`, and estimated SE using `stats.sem(..., ddof=1)`.

In one sentence, distinguish what $s$ and $s/\sqrt{n}$ measure.

**Answer:** $s$ measures the sample's standard deviation (how much the values in the sample deviate from the mean on average), while $s/\sqrt{n}$ measures the standard error of the sample (an estimate of how much the sample mean is expected to vary from the true population mean across random samples).


In [23]:
strength = np.array([
    48.2, 51.4, 49.7, 52.1, 46.8, 50.5, 53.0, 47.6,
    49.1, 51.8, 48.9, 50.7, 47.9, 52.5, 49.8, 50.2,
])

# Calculate and display n, sample_mean, sample_sd, and estimated_se.
n_strength = np.size(strength)
mean_strength = np.sum(strength)/n_strength
std_strength = np.std(strength, ddof = 1)
se_stregnth = stats.sem(strength, ddof = 1)

print("n =",n_strength,
      "\nMean [MPa] =",round(mean_strength,2),
      "\nSample Standard Deviation [MPa] =",round(std_strength,2),
      "\nStandard Error of Sample [MPa] =",round(se_stregnth,2))

n = 16 
Mean [MPa] = 50.01 
Sample Standard Deviation [MPa] = 1.85 
Standard Error of Sample [MPa] = 0.46


### B. Construct and interpret a confidence interval (2 pts)

Because $\sigma$ is unknown, use a **Student's $t$ interval**:

$$\bar x \pm t^*\frac{s}{\sqrt{n}},\qquad df=n-1.$$

Calculate `t_star = stats.t.ppf(0.975, df=n-1)` for 95% confidence. Display $df$, $t^*$, and both interval endpoints in MPa.

Interpret the interval for the **population mean**. Does it describe where 95% of individual strengths fall? Would a 99% interval using the same sample be wider or narrower? No second calculation is required.

**Answer:** The confidence interval does not describe where 95% of individual strengths will fall.
The 95% confidence interval for the population mean strength [MPa] is [ 40.026 , 50.999 ], which means that if we repeated this procedure across many samples, we would expect 95% of those intervals to contain the true population mean. 
Constructing a 99% confidence interval from this sample would result in a larger interval, as in order for 99% of confidence intervals to capture the true population mean, the net cast by each interval would have to be larger.



In [24]:
# Calculate the t critical value and margin of error.
df = n_strength-1
confidence_level = 0.95
cl_input = confidence_level + (1-confidence_level)/2 #CI + alpha/2
t_star = stats.t.ppf(cl_input, df = df)
lower_bound = mean_strength - t_star*se_stregnth
upper_bound = mean_strength + t_star*se_stregnth

# Display df, t_star, and the lower/upper confidence limits in MPa.
print("df =",df,
      "\nt* =",round(t_star,2),
      "\nconfidence interval [MPa] = [",lower_bound,",",upper_bound,"]")


df = 15 
t* = 2.13 
confidence interval [MPa] = [ 49.02574493986514 , 50.999255060134864 ]


## LE7-3. Estimate a population percentage (2 pts)

*Adapted from Chapter 21, Review Exercise 2 (printed p. 391; PDF p. 17).*

In the chapter's historical survey, a simple random sample of **500 households** from a town of **25,000 households** included **239 with internet access**.

Use Python to calculate the estimated proportion and its estimated SE:

$$\hat p=\frac{x}{n},\qquad
\widehat{\operatorname{SE}}(\hat p)=\sqrt{\frac{\hat p(1-\hat p)}{n}}.$$

For this large-sample proportion, use the normal approximation with `z_star = stats.norm.ppf(0.975)`:

$$\hat p\pm z^*\widehat{\operatorname{SE}}(\hat p).$$

Ignore the finite-population correction; the sample includes only 2% of the town's households.

Display the estimated **percentage**, its SE in **percentage points**, and the approximate **95% confidence interval as percentages**. Interpret the interval for all households in the town in one sentence.

**Answer:** The 95% confidence interval for the percentage of housholds in this town with internet is [ 43.42 , 52.18 ], which means that if we repeated this procedure across many samples, we would expect 95% of those intervals to contain the true population percentage. 


In [25]:
total_households = 25_000
sampled_households = 500
with_internet = 239

# Calculate p_hat and its estimated SE using np.sqrt().
p_hat = with_internet/sampled_households
se_households = np.sqrt((p_hat)*(1-p_hat)/sampled_households)

# Find z_star and calculate the lower and upper confidence limits.
z_star = stats.norm.ppf(cl_input)

# Display the estimate, SE, and interval in percentage units.
lower_bound_2 = p_hat - z_star*se_households
upper_bound_2 = p_hat + z_star*se_households

print("estimate for town housholds with internet (%) =",round(p_hat*100,2),
      "\nstandard error of statistic (%) = ",round(se_households*100,2),
      "\nconfidence interval (%) = [",round(lower_bound_2*100,2),",",round(upper_bound_2*100,2),"]")


estimate for town housholds with internet (%) = 47.8 
standard error of statistic (%) =  2.23 
confidence interval (%) = [ 43.42 , 52.18 ]


## Session records

Paste the block from **"make my record"** after each working session. Keep all session records in order.


<!-- Paste your session record blocks below, one after another. -->

*(session records go here)*
{
  "schema_version": "engr-tutor-export-2",
  "assignment": "WA6",
  "export_n": 1,
  "session_date": "2026-10-18",
  "messages": [
    {"i": 1, "who": "student", "text": "I am now working on LE07"},
    {"i": 2, "who": "tutor", "text": "This project is set up for Weekly Assignment 6 (simple linear regression: fitting a line, residuals, diagnostics, and a sensitivity function), so I can't walk you through LE07 here. [trimmed]"},
    {"i": 3, "who": "student", "text": "[uploaded file: LE07.ipynb]"},
    {"i": 4, "who": "tutor", "text": "Thanks for uploading LE07. This project is set up only for Weekly Assignment 6, so I can't tutor the LE07 problems here. [trimmed]"},
    {"i": 5, "who": "student", "text": "Ok I will use you for exclusivley code related questions"},
    {"i": 6, "who": "tutor", "text": "That works. Code questions are exactly what I answer fully and right away. [trimmed]"},
    {"i": 7, "who": "student", "text": "Can you remind me of how to use groupby"},
    {"i": 8, "who": "tutor", "text": "groupby splits a DataFrame into groups, applies a calculation, and combines the results. Toy examples on made-up data covering one or several grouping columns, sum-then-divide ratios, and other summaries. [trimmed]"},
    {"i": 9, "who": "student", "text": "Is there a way to fetch the column titles from a dataframe?"},
    {"i": 10, "who": "tutor", "text": "Yes, the .columns attribute, with toy examples of list(), tolist(), indexing, membership tests, and related attributes. [trimmed]"},
    {"i": 11, "who": "student", "text": "Is there a way to extract the current  column titles and then add an additional title when using pd.concat(keys=)"},
    {"i": 12, "who": "tutor", "text": "Explained how keys= labels pieces, axis=0 versus axis=1, names=, and building a keys list from existing titles, with toy examples. [trimmed]"},
    {"i": 13, "who": "student", "text": "Is there a way to there a way to extract the current  column titles with columns and then add an additional title when using pd.concat(keys=)"},
    {"i": 14, "who": "tutor", "text": "Showed list(df.columns) + [\"new\"] as the keys list for Series pieces, the axis=0 case, and why Index + list does not append. [trimmed]"},
    {"i": 15, "who": "student", "text": "Does keys set the column titles for the new dataframe?"},
    {"i": 16, "who": "tutor", "text": "It depends on Series versus DataFrame pieces and on the axis; toy examples of each case. [trimmed]"},
    {"i": 17, "who": "student", "code": true, "text": "```python\ninspection = pd.DataFrame({\n    \"method\": [\"A\", \"A\", \"B\", \"B\"],\n    \"part_type\": [\"Simple\", \"Complex\", \"Simple\", \"Complex\"],\n    \"inspected\": [20, 80, 80, 20],\n    \"passed\": [18, 48, 64, 10],\n})\n\n# Calculate and display pass rates within each part type.\n# Group by method, sum inspected/passed counts, and calculate overall rates.\npass_rates = inspection.groupby(\"part_type\")[\"passed\"].sum()/inspection.groupby(\"part_type\")[\"inspected\"].sum()\nspecific_pass_rates = inspection[\"passed\"]/inspection[\"inspected\"]\nprint(\"Pass rate (%) for simple parts:\",pass_rates[\"Simple\"]*100,\n\"\\nPass rate (%) for complex parts: \",round(pass_rates[\"Complex\"]*100,2))\n\ninspection_complete = pd.concat([inspection,specific_pass_rates], axis=1, keys = list(inspection.columns) + [\"pass_rate\"])\ninspection_complete.head()\n\n```\n\ninspection = pd.DataFrame({\n    \"method\": [\"A\", \"A\", \"B\", \"B\"],\n    \"part_type\": [\"Simple\", \"Complex\", \"Simple\", \"Complex\"],\n    \"inspected\": [20, 80, 80, 20],\n    \"passed\": [18, 48, 64, 10],\n})\n\n# Calculate and display pass rates within each part type.\n# Group by method, sum inspected/passed counts, and calculate overall rates.\npass_rates = inspection.groupby(\"part_type\")[\"passed\"].sum()/inspection.groupby(\"part_type\")[\"inspected\"].sum()\nspecific_pass_rates = inspection[\"passed\"]/inspection[\"inspected\"]\nprint(\"Pass rate (%) for simple parts:\",pass_rates[\"Simple\"]*100,\n\"\\nPass rate (%) for complex parts: \",round(pass_rates[\"Complex\"]*100,2))\n\ninspection_complete = pd.concat([inspection,specific_pass_rates], axis=1, keys = list(inspection.columns) + [\"pass_rate\"])\ninspection_complete.head()"},
    {"i": 18, "who": "tutor", "text": "Noted no question was included; pointed to the pd.concat line, where two pieces were given five keys, and explained that keys labels pieces rather than columns. Showed assigning a new column as the simpler route, plus notes on rounding and a single groupby with a column list. [trimmed]"},
    {"i": 19, "who": "student", "text": "What command can I use to determine the number of elements in a numpy array?"},
    {"i": 20, "who": "tutor", "text": ".size, len(), and np.size, with toy examples of 1-D versus 2-D arrays and how .size treats np.nan. [trimmed]"},
    {"i": 21, "who": "student", "text": "How can I use stats.sem(..., ddof =1) to get the standard error of a statistic for a sample"},
    {"i": 22, "who": "tutor", "text": "Explained stats.sem as s/sqrt(n) with a toy example, its arguments (ddof, axis, nan_policy), what it returns, and its limits. [trimmed]"},
    {"i": 23, "who": "student", "text": "make my record"}
  ],
  "events": [
    {"i": 7, "part": "none", "type": "explain"},
    {"i": 9, "part": "none", "type": "explain"},
    {"i": 11, "part": "none", "type": "explain"},
    {"i": 13, "part": "none", "type": "explain"},
    {"i": 15, "part": "none", "type": "explain"},
    {"i": 17, "part": "none", "type": "code"},
    {"i": 19, "part": "none", "type": "explain"},
    {"i": 21, "part": "none", "type": "explain"}
  ],
  "parts": [],
  "tutor_assessment": [
    "No Weekly Assignment 6 steps were attempted. The session was general pandas, NumPy, and SciPy code questions (groupby, .columns, pd.concat keys=, array size, stats.sem) from the student's work on a different assignment (LE07). The longest sticking point was pd.concat(keys=), asked three times in different forms, where the student was mixing up keys labeling pieces with labeling columns.",
    "No unstuck or release-valve use occurred. This record covers work outside Weekly Assignment 6, so it does not reflect progress on that assignment."
  ]
}
{
  "schema_version": "engr-tutor-export-2",
  "assignment": "WA6",
  "export_n": 2,
  "session_date": "2026-10-18",
  "messages": [
    {"i": 25, "who": "student", "code": true, "text": "```python\nt_star = stats.t.ppf(0.975, df = df)\n\n```\n\nWhy is 0.975 used for a 95% confidence interval?"},
    {"i": 26, "who": "tutor", "text": "Explained that ppf takes cumulative (left-tail) area, so a two-sided 95% interval leaves 0.025 in each tail and uses 0.975. Gave the general rule ppf((1 + C)/2), a normal-distribution toy check, the one-sided ppf(0.95) mistake, and a note that df = df only works if df holds the degrees of freedom. [trimmed]"},
    {"i": 27, "who": "student", "code": true, "text": "```python\ntotal_households = 25_000\nsampled_households = 500\nwith_internet = 239\n\n# Calculate p_hat and its estimated SE using np.sqrt().\np_hat = with_internet/sampled_households\nse_households = np.sqrt((p_hat)(1-p_hat)/sampled_households)\n\n# Find z_star and calculate the lower and upper confidence limits.\n# Display the estimate, SE, and interval in percentage units.\n\n```\n\n---------------------------------------------------------------------------\nTypeError                                 Traceback (most recent call last)\nCell In[53], line 7\n      3 with_internet = 239\n      4 \n      5 # Calculate p_hat and its estimated SE using np.sqrt().\n      6 p_hat = with_internet/sampled_households\n----> 7 se_households = np.sqrt((p_hat)(1-p_hat)/sampled_households)\n      8 \n      9 # Find z_star and calculate the lower and upper confidence limits.\n     10 # Display the estimate, SE, and interval in percentage units.\n\nTypeError: 'float' object is not callable"},
    {"i": 28, "who": "tutor", "text": "Explained that parentheses after a value mean a function call in Python, so (p_hat)(1-p_hat) raises 'float' object is not callable and an explicit * is needed. Gave a toy example, noted the SE is a proportion so percentage conversion belongs in the display, and offered to explain stats.norm.ppf. [trimmed]"},
    {"i": 29, "who": "student", "text": "make my record"}
  ],
  "events": [
    {"i": 25, "part": "none", "type": "explain"},
    {"i": 27, "part": "none", "type": "code"}
  ],
  "parts": [],
  "tutor_assessment": [
    "No Weekly Assignment 6 steps were attempted. This session covered general code and statistics questions from the student's LE07 work: why 0.975 is used for a 95% two-sided t interval, and a 'float' object is not callable TypeError caused by a missing * in the standard error formula. The sticking point was Python's requirement for an explicit multiplication operator.",
    "No unstuck or release-valve use occurred. This record covers work outside Weekly Assignment 6, so it does not reflect progress on that assignment."
  ]
}
{
  "schema_version": "engr-tutor-export-2",
  "assignment": "WA7",
  "export_n": 1,
  "session_date": "2026-10-20",
  "messages": [
    {"i": 1, "who": "student", "text": "Hello! Can you quickly check my work on the questions from the assignment? I used LE6 for help on commands and want to make sure I didnt miss anything.\n\n```python\ninspection = pd.DataFrame({\n    \"method\": [\"A\", \"A\", \"B\", \"B\"],\n    \"part_type\": [\"Simple\", \"Complex\", \"Simple\", \"Complex\"],\n    \"inspected\": [20, 80, 80, 20],\n    \"passed\": [18, 48, 64, 10],\n})\n\n# Calculate and display pass rates within each part type.\n# Group by method, sum inspected/passed counts, and calculate overall rates.\npass_rates = inspection.groupby(\"method\")[\"passed\"].sum()/inspection.groupby(\"method\")[\"inspected\"].sum()\nspecific_pass_rates = inspection[\"passed\"]/inspection[\"inspected\"]\nprint(\"Pass rate (%) for method A:\",pass_rates[\"A\"]*100,\n\"\\nPass rate (%) for method B: \",round(pass_rates[\"B\"]*100,2))\n\ninspection_complete = pd.concat([inspection,specific_pass_rates], axis=1, keys = [\"Inspection Data\"] + [\"Pass Rate\"])\ninspection_complete.head()\n\n```", "code": true},
    {"i": 2, "who": "tutor", "text": "Your LE7-1 code is doing what the prompt asks. The overall rates come from `groupby(\"method\")` sums of `passed` and `inspected`, divided after summing, not from averaged percentages. [trimmed] Notes on display, rounding, and the empty Answer field. Two starter questions: how do each method's 100 inspections split between simple and complex parts, and which part type passes more often under each method?"},
    {"i": 3, "who": "student", "text": "```markdown\n**Answer:** Method B has the higher overall rate; however method A has a higher pass rate within simple and complex parts. This reverse effect is illustrative of Simpson's Paradox. Simple parts from this sample have a higher pass rate than complex parts, and since more simple parts were inspected using method B than method A (and more complex parts were inspected using method A than method B), the overall pass rate for method B is higher than method A. These observations do not establsh a causal effect of method because the experiment did not involve random assignment of methods to parts.\n\n```"},
    {"i": 4, "who": "tutor", "text": "Your factual statements check out against my own calculation of the counts. [trimmed] Pointed out the explanation runs longer than two sentences, asked what kind of data these are and what else could differ between the parts each method received, and noted a spelling slip."},
    {"i": 5, "who": "student", "text": "```markdown\n**Answer:** Method B has the higher overall rate; however method A has a higher pass rate within simple and complex parts. This reverse effect is illustrative of Simpson's Paradox. Simple parts from this sample have a higher pass rate than complex parts, and since more simple parts were inspected using method B than method A (and more complex parts were inspected using method A than method B), the overall pass rate for method B is higher than method A. These observations do not establish a causal effect of method because the experiment did not involve random assignment of methods to parts, thus resulting in Simpson's Paradox being present in the data.\n\n```"},
    {"i": 6, "who": "tutor", "text": "Thanks for fixing the spelling. [trimmed] Asked whether the lack of random assignment really causes the reversal or whether group composition does, asked again what else could differ between the parts each method received, and noted the length is still above two sentences."},
    {"i": 7, "who": "student", "text": "```python\nstrength = np.array([\n    48.2, 51.4, 49.7, 52.1, 46.8, 50.5, 53.0, 47.6,\n    49.1, 51.8, 48.9, 50.7, 47.9, 52.5, 49.8, 50.2,\n])\n\n# Calculate and display n, sample_mean, sample_sd, and estimated_se.\nn_strength = np.size(strength)\nmean_strength = np.sum(strength)/n_strength\nstd_strength = np.std(strength, ddof = 1)\nse_stregnth = stats.sem(strength, ddof = 1)\n\nprint(\"n =\",n_strength,\n      \"\\nMean [MPa] =\",round(mean_strength,2),\n      \"\\nSample Standard Deviation [MPa] =\",round(std_strength,2),\n      \"\\nStandard Error of Sample [MPa] =\",round(se_stregnth,2))\n```\n\n\n```markdown\n**Answer:** $s$ measures the sample's standard deviation (how much the values in the sample deviate from the mean on average), while $s/\\sqrt{n}$ measures the standard error of the sample (an estimate of how much the sample mean is expected to vary from the true population mean across random samples).\n\n```", "code": true},
    {"i": 8, "who": "tutor", "text": "Your code does what part A asks. [trimmed] Noted the variable names and the need to re-run this cell before LE7-2B, and asked what s is the spread of and which of the two shrinks as more parts are tested."},
    {"i": 9, "who": "student", "text": "```python\n# Calculate the t critical value and margin of error.\ndf = n_strength-1\nconfidence_level = 0.95\ncl_input = confidence_level + (1-confidence_level)/2 #CI + alpha/2\nt_star = stats.t.ppf(cl_input, df = df)\nlower_bound = mean_strength - t_star*se_stregnth\nupper_bound = mean_strength + t_star*se_stregnth\n\n# Display df, t_star, and the lower/upper confidence limits in MPa.\nprint(\"df =\",df,\n      \"\\nt* =\",round(t_star,2),\n      \"\\nconfidence interval [MPa] = [\",lower_bound,\",\",upper_bound,\"]\")\n```\n\n\n```markdown\n**Answer:** The confidence interval does not describe where 95% of individual strengths will fall.\nThe 95% confidence interval for the population mean strength [MPa] is [ 40.026 , 50.999 ], which means that if we repeated this procedure across many samples, we would expect 95% of those intervals to contain the true population mean. \nConstructing a 99% confidence interval from this sample would result in a larger interval, as in order for 99% of confidence intervals to capture the true population mean, the net cast by each interval would have to be larger.\n```", "code": true},
    {"i": 10, "who": "tutor", "text": "Your code for part B follows the prompt. [trimmed] Asked the student to compare the typed interval with their printed output, noted the `df` naming, and asked why the interval does not describe individual strengths (width versus spread) and which number in the formula changes at 99%."},
    {"i": 11, "who": "student", "text": "```python\ntotal_households = 25_000\nsampled_households = 500\nwith_internet = 239\n\n# Calculate p_hat and its estimated SE using np.sqrt().\np_hat = with_internet/sampled_households\nse_households = np.sqrt((p_hat)*(1-p_hat)/sampled_households)\n\n# Find z_star and calculate the lower and upper confidence limits.\nz_star = stats.norm.ppf(cl_input)\n\n# Display the estimate, SE, and interval in percentage units.\nlower_bound_2 = p_hat - z_star*se_households\nupper_bound_2 = p_hat + z_star*se_households\n\nprint(\"estimate for town housholds with internet (%) =\",round(p_hat*100,2),\n      \"\\nstandard error of statistic (%) = \",round(se_households*100,2),\n      \"\\nconfidence interval (%) = [\",round(lower_bound_2*100,2),\",\",round(upper_bound_2*100,2),\"]\")\n```\n\n\n```markdown\n**Answer:** The 95% confidence interval for the percentage of housholds in this town with internet is [ 43.42 , 52.18 ], which means that if we repeated this procedure across many samples, we would expect 95% of those intervals to contain the true population percentage. \n\n```", "code": true},
    {"i": 12, "who": "tutor", "text": "Your LE7-3 code follows the prompt. [trimmed] Noted that `z_star` depends on `cl_input` from the LE7-2B cell, that the SE label should say percentage points, and asked the student to compare typed intervals (here and in part B) with printed output. On the sentence, asked which population it describes and whether it should include the point estimate."}
  ],
  "events": [
    {"i": 1, "part": "7-1", "type": "code"},
    {"i": 7, "part": "7-2A", "type": "code"},
    {"i": 9, "part": "7-2B", "type": "code"},
    {"i": 11, "part": "7-3", "type": "code"}
  ],
  "parts": [
    {"part": "7-1", "exchanges": 3, "max_tier": 1, "passed": false},
    {"part": "7-2A", "exchanges": 1, "max_tier": 1, "passed": false},
    {"part": "7-2B", "exchanges": 1, "max_tier": 1, "passed": false},
    {"part": "7-3", "exchanges": 1, "max_tier": 1, "passed": false}
  ],
  "tutor_assessment": [
    "The student spent the most time on the LE7-1 written explanation (three exchanges): the causal sentence stayed focused on missing random assignment rather than on what other differences the records cannot rule out, and the explanation ran past two sentences. In LE7-2B and LE7-3 the typed interval endpoints were flagged for checking against printed output, and the prose answers there were still being refined when the session ended.",
    "No unstuck or release-valve use occurred, and nothing looked like a tutoring failure. All code was student-written and needed only minor checks; the open work at the end of the session is the written answers."
  ]
}

## Before you submit

1. **Restart Kernel and Run All Cells**; confirm there are no errors.
2. Check that calculations, tables, and written answers are visible.
3. Include your session record block(s).
4. Save and submit the completed `.ipynb` file in Canvas.
